# RunMode

This is the core value proposition of the framework: a single agent can operate deterministically *and* autonomously, switching between modes at runtime. When a workflow step fails, the agent automatically falls back to LLM reasoning to fix the problem, then seamlessly resumes the workflow.

In this tutorial, we'll build a **form automation agent** — it normally fills forms step-by-step (Workflow mode), but when it encounters unexpected CAPTCHAs or popups, it switches to Agent mode to handle them, then resumes the workflow.

## Initialize

First, let's set up the LLM and define the tools our form automation agent will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = LLM(model="openai/gpt-4o-mini")

In [ ]:
from bridgic.core import tool

attempt_counter = {"captcha_attempts": 0}

@tool(description="Fill a form field with the given value")
async def fill_field(field_name: str, value: str) -> str:
    return f"Filled '{field_name}' with '{value}'"

@tool(description="Click a button on the page")
async def click_button(button_name: str) -> str:
    if button_name == "submit" and attempt_counter["captcha_attempts"] == 0:
        attempt_counter["captcha_attempts"] += 1
        raise RuntimeError("CAPTCHA detected! Cannot submit directly.")
    return f"Clicked '{button_name}' successfully"

@tool(description="Solve a CAPTCHA challenge on the page")
async def solve_captcha() -> str:
    return "CAPTCHA solved successfully"

@tool(description="Check the current page status")
async def check_page_status() -> str:
    if attempt_counter["captcha_attempts"] > 0:
        return "Page status: CAPTCHA challenge displayed, form fields are filled"
    return "Page status: Form ready for input"

We have four tools:

- **`fill_field`** — fills a form field with a given value.
- **`click_button`** — clicks a button. On the first submit attempt, it simulates a CAPTCHA challenge by raising a `RuntimeError`.
- **`solve_captcha`** — solves the CAPTCHA challenge.
- **`check_page_status`** — reports the current state of the page.

The `attempt_counter` dictionary simulates a real-world scenario: the first time we try to submit the form, a CAPTCHA appears. After solving it, subsequent submissions succeed.

## Part 1: The Four Run Modes

Bridgic Amphibious provides four `RunMode` options that control how your agent executes:

| Mode | Behavior |
|------|----------|
| **`RunMode.AGENT`** | Pure agent mode — only `on_agent()` runs. The LLM decides every step. |
| **`RunMode.WORKFLOW`** | Pure workflow mode — only `on_workflow()` runs. If any step errors, the entire workflow fails. |
| **`RunMode.AMPHIBIOUS`** | Workflow first, automatic agent fallback on failure. The best of both worlds. |
| **`RunMode.AUTO`** (default) | Auto-detects based on whether `on_workflow()` is defined. If it is, behaves like `AMPHIBIOUS`; otherwise, behaves like `AGENT`. |

Let's define the agent that we'll run in different modes.

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker,
    think_unit, step, RunMode
)

class FormFiller(AmphibiousAutoma[CognitiveContext]):
    fixer = think_unit(
        CognitiveWorker.inline(
            "Something went wrong during the form submission. "
            "Check the page status, identify the problem, and fix it."
        ),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.fixer

    async def on_workflow(self, ctx: CognitiveContext):
        yield step("fill_field", field_name="username", value="john_doe")
        yield step("fill_field", field_name="password", value="s3cur3p@ss")
        yield step("fill_field", field_name="email", value="john@example.com")
        yield step("click_button", button_name="submit")

Our `FormFiller` agent has both orchestration methods:

- **`on_workflow()`** defines a deterministic, step-by-step form-filling process: fill three fields, then click submit.
- **`on_agent()`** activates a `CognitiveWorker` that diagnoses and fixes problems using LLM reasoning.

The `RunMode` we choose determines which method (or both) gets used.

### Pure Workflow Mode — Fails on Error

Let's first run in `RunMode.WORKFLOW` to see what happens when the CAPTCHA appears.

In [ ]:
# Reset state
attempt_counter["captcha_attempts"] = 0

try:
    agent = FormFiller(llm=llm)
    result = await agent.arun(
        goal="Fill out the registration form",
        tools=[fill_field, click_button, solve_captcha, check_page_status],
        mode=RunMode.WORKFLOW,
    )
    print(result)
except Exception as e:
    print(f"Workflow failed: {e}")

In pure Workflow mode, when `click_button("submit")` hits the CAPTCHA error, the entire workflow fails. There is no recovery mechanism — the agent simply raises the exception and stops. The three fields were filled successfully, but the submission could not be completed.

This is the fundamental limitation of a purely deterministic workflow: it can only handle the happy path.

## Part 2: Amphibious Mode — Automatic Fallback & Recovery

Now let's see the magic of Amphibious mode. Here is how the automatic fallback mechanism works:

1. **Workflow steps execute normally** — each `yield step(...)` runs in order.
2. **When a step fails** — the framework catches the error and activates `on_agent()`.
3. **The LLM analyzes the problem** — it uses tools to diagnose and fix the issue.
4. **After fixing** — the workflow resumes from the next step.

This is the "amphibious" design: the efficiency of deterministic workflows for the happy path, combined with the adaptability of LLM reasoning for unexpected situations.

In [ ]:
# Reset state
attempt_counter["captcha_attempts"] = 0

agent = FormFiller(llm=llm)
result = await agent.arun(
    goal="Fill out the registration form",
    tools=[fill_field, click_button, solve_captcha, check_page_status],
    mode=RunMode.AMPHIBIOUS,
    will_fallback=True,
    max_consecutive_fallbacks=2,
)
print(result)

Now when `click_button("submit")` fails, the framework automatically activates `on_agent()`. The LLM discovers the CAPTCHA, calls `solve_captcha()`, then re-attempts the submit. The workflow continues seamlessly.

Two key parameters control the fallback behavior:

- **`will_fallback=True`**: Enables the automatic agent fallback mechanism. Without this, Amphibious mode behaves the same as Workflow mode.
- **`max_consecutive_fallbacks=2`**: A safety valve — if 2 consecutive steps fail and the agent cannot fix them, the framework abandons the workflow entirely and runs in pure Agent mode. This prevents infinite fallback loops.

## Part 3: Comparing All Three Modes

Let's see how the same task plays out under each mode, side by side.

In [ ]:
for mode_name, mode in [("AGENT", RunMode.AGENT), ("WORKFLOW", RunMode.WORKFLOW), ("AMPHIBIOUS", RunMode.AMPHIBIOUS)]:
    attempt_counter["captcha_attempts"] = 0
    print(f"\n{'='*50}")
    print(f"Running in {mode_name} mode")
    print(f"{'='*50}")

    try:
        agent = FormFiller(llm=llm)
        result = await agent.arun(
            goal="Fill out the registration form",
            tools=[fill_field, click_button, solve_captcha, check_page_status],
            mode=mode,
            will_fallback=True,
            max_consecutive_fallbacks=2,
        )
        print(f"Result: {result}")
    except Exception as e:
        print(f"Failed: {e}")

Here is a summary of how the three modes compare:

| Mode | CAPTCHA Handling | Efficiency | Predictability |
|------|-----------------|------------|----------------|
| **WORKFLOW** | Fails immediately | High (no LLM overhead) | Highest (but fragile) |
| **AGENT** | LLM figures it out | Lower (LLM reasons every step) | Lower (may vary) |
| **AMPHIBIOUS** | Fixed steps + LLM rescue | Best of both | High with flexibility |

- **WORKFLOW** is the fastest and most predictable, but it cannot handle anything unexpected. It is ideal for stable, well-understood processes with no surprises.
- **AGENT** can handle anything the LLM can reason about, but it is slower and less predictable because the LLM decides every single step.
- **AMPHIBIOUS** gives you the best of both worlds: deterministic efficiency on the happy path, with automatic LLM-powered recovery when things go wrong.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we saw the amphibious design in action:

- **RunMode** controls how the agent runs: `AGENT` (LLM only), `WORKFLOW` (deterministic only), `AMPHIBIOUS` (workflow + agent fallback), or `AUTO` (auto-detect).
- **Amphibious mode** is the framework's core innovation: workflow steps execute deterministically, but when a step fails, the framework automatically activates `on_agent()` to let the LLM diagnose and fix the problem.
- **`will_fallback=True`** enables the automatic fallback mechanism. **`max_consecutive_fallbacks`** controls how many consecutive failures trigger a full switch to agent mode.
- The amphibious design gives you the **efficiency of workflows** for the happy path and the **adaptability of agents** for unexpected situations.

Next, we'll explore Context and Exposure — how to control what information the LLM sees when making decisions.